# Model Evaluation — COMP0047 Data Science Project

**Author:** Yuqiu He (Jack) — TinsureH  
**Date:** 2026-03-20

## Purpose

This notebook provides a standardised evaluation of all prediction models built in this project:
- **Logistic Regression** (basecases: 1-day, 5-day, 20-day)
- **XGBoost** (basecases: 1-day, 5-day, 20-day)
- **PatchTST** (lags: 1, 5, 20)
- **LSTM** (lags: 1, 5, 30 — aggregated metrics only)

## ⚠️ Known Alignment Issue (Rahil's Model Alignment Notes)

The models are currently **not evaluated on the same time period**, which makes direct comparison unreliable:

| Model | Evaluated On | Date Range |
|-------|-------------|------------|
| Logistic / XGBoost | **Validation set** (15%) | 2016-07 – 2021-04 |
| PatchTST | **Test set** (15%) | 2021-06 – 2026-02 |
| LSTM | **Test set** (aggregated only) | — |

The canonical split (per the shared Google Doc) is:
```
train_end = int(total_rows * 0.70)  # → 2016-06-29
val_end   = int(total_rows * 0.85)  # → 2021-04-21
```

**Proposed fix (Stephan + Rahil):** Re-run Logistic/XGBoost on the **test set** to align with PatchTST. LSTM needs to be re-run with lag 20 (not 30) and must output row-level predictions. Until then, this notebook evaluates each model on its available predictions and flags the discrepancy.

## Evaluation Strategy

We adopt a **risk-averse** stance: missing a bear market (False Negative for bear class) is more costly than a false alarm. Therefore:
- Primary metric: **F2-score** (β=2 → Recall weighted 2× over Precision)
- Secondary metrics: Accuracy, Recall (Bear), Precision (Bear), MCC

## 1. Setup

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    fbeta_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_auc_score
)

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT   = Path(os.getcwd()).parents[1]          # notebooks/07 → repo root
REPORTS_DIR = REPO_ROOT / 'reports'
DATA_DIR    = REPO_ROOT / 'data'
OUTPUT_DIR  = REPORTS_DIR / 'evaluation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = {'logistic_regression': '#4C72B0', 'xgboost': '#DD8452',
           'patchtst': '#55A868', 'lstm': '#C44E52'}

print('Repo root:', REPO_ROOT)
print('Output  :', OUTPUT_DIR)

Repo root: /Users/rahilshah/Development
Output  : /Users/rahilshah/Development/reports/evaluation


## 2. Load & Standardise Predictions

In [2]:
def load_predictions(path: Path, model: str, horizon: str, split: str) -> pd.DataFrame:
    """Load a predictions CSV and normalise columns to the project standard."""
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.rename(columns={'Date': 'prediction_date'})

    # Standardise required columns
    if 'model'   not in df.columns: df['model']   = model
    if 'horizon' not in df.columns: df['horizon']  = horizon
    if 'split'   not in df.columns: df['split']    = split

    # Keep only standard columns + drop PatchTST's GSPC
    keep = ['prediction_date', 'y_true', 'y_pred', 'y_pred_prob', 'model', 'horizon', 'split']
    df = df[[c for c in keep if c in df.columns]]
    df['y_true'] = df['y_true'].astype(int)
    df['y_pred'] = df['y_pred'].astype(int)
    return df


# ── Logistic Regression ────────────────────────────────────────────────────
logistic_dfs = [
    load_predictions(REPORTS_DIR/'trained_logistic/basecase_1day/predictions.csv',
                     'logistic_regression', '1day', 'validation'),
    load_predictions(REPORTS_DIR/'trained_logistic/basecase_5days/predictions.csv',
                     'logistic_regression', '5day', 'validation'),
    load_predictions(REPORTS_DIR/'trained_logistic/basecase_20days/predictions.csv',
                     'logistic_regression', '20day', 'validation'),
]

# ── XGBoost ───────────────────────────────────────────────────────────────
xgb_dfs = [
    load_predictions(REPORTS_DIR/'trained_xgboost/basecase_1day/predictions.csv',
                     'xgboost', '1day', 'validation'),
    load_predictions(REPORTS_DIR/'trained_xgboost/basecase_5days/predictions.csv',
                     'xgboost', '5day', 'validation'),
    load_predictions(REPORTS_DIR/'trained_xgboost/basecase_20days/predictions.csv',
                     'xgboost', '20day', 'validation'),
]

# ── PatchTST ──────────────────────────────────────────────────────────────
patchtst_dfs = [
    load_predictions(REPORTS_DIR/'trained_patchtst/results_lag_1/predictions.csv',
                     'patchtst', '1day', 'test'),
    load_predictions(REPORTS_DIR/'trained_patchtst/results_lag_5/predictions.csv',
                     'patchtst', '5day', 'test'),
    load_predictions(REPORTS_DIR/'trained_patchtst/results_lag_20/predictions.csv',
                     'patchtst', '20day', 'test'),
]

# ── Combine all row-level predictions ─────────────────────────────────────
all_preds = pd.concat(logistic_dfs + xgb_dfs + patchtst_dfs, ignore_index=True)

print(f'Total prediction rows loaded: {len(all_preds):,}')
print()
print(all_preds.groupby(['model', 'horizon', 'split']).agg(
    rows=('y_true', 'count'),
    date_from=('prediction_date', 'min'),
    date_to=('prediction_date', 'max')
).to_string())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/rahilshah/Development/reports/trained_logistic/basecase_1day/predictions.csv'

### 2.1 Load LSTM Aggregated Metrics (no row-level predictions available)

> **Note:** The LSTM currently only outputs aggregated test metrics. Row-level predictions are needed for fair comparison. Additionally, LSTM was trained with **lag 30** — it should be re-run with **lag 20** to match other models. See [Rahil's alignment notes].

In [ ]:
def load_lstm_metrics(path: Path, horizon: str) -> dict:
    df = pd.read_csv(path).rename(columns={'Unnamed: 0': 'dataset'})
    row = df[df['dataset'] == 'Test Set'].iloc[0]
    return {
        'model': 'lstm', 'horizon': horizon, 'split': 'test',
        'accuracy': round(row['accuracy'], 4),
        'roc_auc':  round(row['roc_auc'],  4),
        'precision': round(row['precision'], 4),
        'recall':    round(row['recall'],    4),
        'note': 'Aggregated only — no row-level predictions. Re-run needed (lag 20).'
    }

lstm_metrics_agg = pd.DataFrame([
    load_lstm_metrics(REPORTS_DIR/'trained_ltsm/results_lag_1/test_metrics.csv',  '1day'),
    load_lstm_metrics(REPORTS_DIR/'trained_ltsm/results_lag_5/test_metrics.csv',  '5day'),
    load_lstm_metrics(REPORTS_DIR/'trained_ltsm/results_lag_30/test_metrics.csv', '20day*'),
])

print('LSTM aggregated metrics (test set):')
display(lstm_metrics_agg)

## 3. Metrics Computation

In [ ]:
def compute_metrics(df: pd.DataFrame, beta: float = 2.0) -> dict:
    """
    Compute the full evaluation metrics for a binary Bear(0)/Bull(1) classifier.
    Risk-averse: Bear recall is prioritised via F-beta (beta=2).
    """
    y_true = df['y_true'].values
    y_pred = df['y_pred'].values
    y_prob = df['y_pred_prob'].values if 'y_pred_prob' in df.columns else None

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        # Overall
        'accuracy':          round(accuracy_score(y_true, y_pred), 4),
        # Bear-class (label=0) — risk-averse priority
        'bear_precision':    round(precision_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'bear_recall':       round(recall_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'bear_f2':           round(fbeta_score(y_true, y_pred, beta=beta, pos_label=0, zero_division=0), 4),
        # Bull-class (label=1)
        'bull_precision':    round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'bull_recall':       round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'bull_f2':           round(fbeta_score(y_true, y_pred, beta=beta, pos_label=1, zero_division=0), 4),
        # Macro F2 (overall balance)
        'macro_f2':          round(fbeta_score(y_true, y_pred, beta=beta, average='macro', zero_division=0), 4),
        # MCC — robust to class imbalance
        'mcc':               round(matthews_corrcoef(y_true, y_pred), 4),
        # ROC-AUC (needs probabilities)
        'roc_auc':           round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else None,
        # Confusion matrix raw counts
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'n_samples': len(y_true),
    }
    return metrics


# ── Compute for every model × horizon combination ─────────────────────────
results = []
for (model, horizon, split), grp in all_preds.groupby(['model', 'horizon', 'split']):
    m = compute_metrics(grp)
    m.update({'model': model, 'horizon': horizon, 'split': split})
    results.append(m)

results_df = pd.DataFrame(results)

# Reorder columns for readability
col_order = ['model', 'horizon', 'split', 'n_samples',
             'accuracy', 'bear_recall', 'bear_precision', 'bear_f2',
             'macro_f2', 'mcc', 'roc_auc',
             'tn', 'fp', 'fn', 'tp']
results_df = results_df[col_order].sort_values(['horizon', 'model'])

print('=== Evaluation Results (row-level models) ===')
display(results_df.set_index(['model', 'horizon']))

## 4. Summary Comparison Table

In [ ]:
# Pivot: models as rows, horizons as columns — primary metric = bear_f2
pivot = results_df.pivot_table(
    index='model', columns='horizon',
    values=['accuracy', 'bear_recall', 'bear_f2', 'macro_f2', 'mcc', 'roc_auc']
)
pivot.columns = [f'{m}_{h}' for m, h in pivot.columns]

print('=== Summary Pivot Table ===')
display(pivot.round(4))

# Save
results_df.to_csv(OUTPUT_DIR / 'metrics_per_model_horizon.csv', index=False)
pivot.to_csv(OUTPUT_DIR / 'metrics_pivot.csv')
print(f'\nSaved to {OUTPUT_DIR}')

## 5. Visualisations
### 5.1 Bear F2-Score Comparison (per horizon)

In [ ]:
horizons  = ['1day', '5days', '20days']
metrics_to_plot = [
    ('bear_f2',     'Bear F2-Score (β=2)',  'Primary metric — risk-averse'),
    ('bear_recall', 'Bear Recall',           'Must not miss bear markets'),
    ('accuracy',    'Accuracy',              'Overall correctness'),
    ('mcc',         'MCC',                   'Robust to class imbalance'),
]

fig, axes = plt.subplots(len(metrics_to_plot), len(horizons),
                         figsize=(14, 4 * len(metrics_to_plot)), sharey='row')
fig.suptitle('Model Comparison — All Metrics × Horizons\n'
             '⚠️  Note: Logistic/XGBoost on validation set; PatchTST on test set',
             fontsize=13, y=1.01)

for row_idx, (metric, metric_label, subtitle) in enumerate(metrics_to_plot):
    for col_idx, horizon in enumerate(horizons):
        ax = axes[row_idx][col_idx]
        sub = results_df[results_df['horizon'] == horizon].copy()
        colors = [PALETTE.get(m, '#888') for m in sub['model']]

        bars = ax.bar(sub['model'], sub[metric], color=colors, width=0.5, alpha=0.88)
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)

        ax.set_title(f'{horizon}\n{subtitle}', fontsize=9, color='#555')
        ax.set_ylim(0, 1.12)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=15, labelsize=8)

        if col_idx == 0:
            ax.set_ylabel(metric_label, fontsize=9)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'model_comparison_all_metrics.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {OUTPUT_DIR / "model_comparison_all_metrics.png"}')

### 5.2 Confusion Matrices — All Models × Horizons

In [ ]:
models   = ['logistic_regression', 'xgboost', 'patchtst']
horizons = ['1day', '5days', '20days']

fig, axes = plt.subplots(len(models), len(horizons),
                         figsize=(13, 4 * len(models)))
fig.suptitle('Confusion Matrices — Bear(0) / Bull(1)\n'
             '⚠️  Logistic/XGBoost on validation; PatchTST on test',
             fontsize=12, y=1.01)

for ri, model in enumerate(models):
    for ci, horizon in enumerate(horizons):
        ax = axes[ri][ci]
        sub = all_preds[(all_preds['model'] == model) &
                        (all_preds['horizon'] == horizon)]
        if sub.empty:
            ax.axis('off')
            continue

        cm = confusion_matrix(sub['y_true'], sub['y_pred'], labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt='d', ax=ax, cbar=False,
                    cmap='Blues',
                    xticklabels=['Bear(0)', 'Bull(1)'],
                    yticklabels=['Bear(0)', 'Bull(1)'])

        # Highlight False Negatives (bear predicted as bull) in red text
        ax.texts[1].set_color('red')   # top-right = FN for bear

        row = results_df[(results_df['model'] == model) &
                         (results_df['horizon'] == horizon)]
        f2  = row['bear_f2'].values[0]  if len(row) else '—'
        rec = row['bear_recall'].values[0] if len(row) else '—'
        ax.set_title(f'{model}\n{horizon} | Bear-F2={f2:.3f}  Recall={rec:.3f}',
                     fontsize=8)
        ax.set_xlabel('Predicted', fontsize=8)
        ax.set_ylabel('Actual', fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrices_all.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {OUTPUT_DIR / "confusion_matrices_all.png"}')

### 5.3 Prediction Timeline — How models classify regimes over time

In [ ]:
fig, axes = plt.subplots(len(models), 1, figsize=(14, 3.5 * len(models)), sharex=False)
fig.suptitle('Prediction Timeline (1-day horizon) — Bear vs Bull Calls',
             fontsize=12)

for ax, model in zip(axes, models):
    sub = all_preds[(all_preds['model'] == model) & (all_preds['horizon'] == '1day')].copy()
    sub = sub.sort_values('prediction_date')

    # Shade actual bear regions
    bear_mask = sub['y_true'] == 0
    prev = False
    for i, (idx, row) in enumerate(sub.iterrows()):
        cur = bool(bear_mask[idx])
        if cur and not prev:
            start = row['prediction_date']
        elif not cur and prev:
            ax.axvspan(start, row['prediction_date'], alpha=0.15, color='red', label='_')
        prev = cur
    if prev:
        ax.axvspan(start, sub['prediction_date'].iloc[-1], alpha=0.15, color='red')

    # Plot predicted probability
    ax.plot(sub['prediction_date'], sub['y_pred_prob'], lw=0.7,
            color=PALETTE.get(model, '#555'), alpha=0.8, label='P(Bull)')
    ax.axhline(0.5, ls='--', lw=0.8, color='grey', label='Threshold=0.5')

    split_label = sub['split'].iloc[0]
    ax.set_title(f'{model} ({split_label} set) — red shading = actual bear', fontsize=9)
    ax.set_ylabel('P(Bull)', fontsize=8)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'prediction_timelines.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {OUTPUT_DIR / "prediction_timelines.png"}')

## 6. LSTM — Aggregated Metrics Only

> LSTM does not currently produce row-level predictions. The following are the aggregated test-set metrics. The model was trained with **lag 30** (should be lag 20 per alignment agreement). Re-run needed.

In [ ]:
print('=== LSTM Test Metrics (aggregated) ===')
display(lstm_metrics_agg[['model', 'horizon', 'split', 'accuracy', 'roc_auc', 'precision', 'recall', 'note']])

# Save
lstm_metrics_agg.to_csv(OUTPUT_DIR / 'lstm_aggregated_metrics.csv', index=False)
print(f'Saved → {OUTPUT_DIR / "lstm_aggregated_metrics.csv"}')

## 7. Cross-Validation Results

In [ ]:
def load_cv_test_metrics(path: Path, model: str, horizon: str) -> dict | None:
    """Load cross-validation test metrics if available."""
    try:
        df = pd.read_csv(path)
        # Some files have an unnamed index column
        if 'Unnamed: 0' in df.columns:
            df = df.rename(columns={'Unnamed: 0': 'dataset'})
        row = df.iloc[0] if 'dataset' not in df.columns else df[df['dataset'] == 'Test Set'].iloc[0]
        return {'model': model, 'horizon': horizon, **row.to_dict()}
    except Exception:
        return None

cv_rows = [
    load_cv_test_metrics(REPORTS_DIR/'trained_logistic/cross_validation/test_metrics.csv',
                         'logistic_regression', 'cv'),
    load_cv_test_metrics(REPORTS_DIR/'trained_xgboost/cross_validation/test_metrics.csv',
                         'xgboost', 'cv'),
]
cv_df = pd.DataFrame([r for r in cv_rows if r])
print('=== Cross-Validation Test Metrics ===')
display(cv_df)

## 8. Full Classification Report

In [ ]:
reports = []
for (model, horizon), grp in all_preds.groupby(['model', 'horizon']):
    report = classification_report(
        grp['y_true'], grp['y_pred'],
        target_names=['Bear (0)', 'Bull (1)'],
        output_dict=True, zero_division=0
    )
    print(f'\n── {model.upper()} | horizon={horizon} | n={len(grp)} ──')
    print(classification_report(
        grp['y_true'], grp['y_pred'],
        target_names=['Bear (0)', 'Bull (1)'], zero_division=0
    ))

## 9. Key Findings & Action Items

### Findings

Summarise after running the notebook — fill in based on actual numbers.

### ⚠️ Action Items (before final evaluation is valid)

| Priority | Task | Owner | Status |
|----------|------|-------|--------|
| 🔴 High | Re-run Logistic & XGBoost on **test set** (2021-04 – 2026-02) | Stephan / Jack | Pending |
| 🔴 High | Re-run LSTM with **lag 20** and output row-level predictions | Ben | Pending |
| 🟡 Med  | Align PatchTST prediction start date (sequence length offset) | Yajie | Pending |
| 🟡 Med  | Add MCC and F2 to all existing model output CSVs | All | Pending |
| 🟢 Low  | Add Bayesian / ablation test for feature importance | — | Future |

Once Logistic/XGBoost test-set predictions are available, replace the validation-set CSV paths in **Section 2** with the test-set equivalents and re-run this notebook.

In [ ]:
# Print a clean summary for the group chat / shared doc
print('='*60)
print('MODEL EVALUATION SUMMARY')
print('='*60)
print(f'Evaluated {len(results_df)} model×horizon combinations')
print(f'Metric priority: Bear F2-score (β=2) — risk averse\n')

for horizon in ['1day', '5days', '20days']:
    sub = results_df[results_df['horizon'] == horizon].sort_values('bear_f2', ascending=False)
    print(f'Horizon: {horizon}')
    for _, r in sub.iterrows():
        flag = ' ⚠️ val-set' if r['split'] == 'validation' else ' ✓ test-set'
        print(f"  {r['model']:25s} Bear-F2={r['bear_f2']:.3f}  "
              f"Recall={r['bear_recall']:.3f}  Acc={r['accuracy']:.3f}  MCC={r['mcc']:.3f}{flag}")
    print()

print('LSTM (aggregated test metrics only — not directly comparable):')
for _, r in lstm_metrics_agg.iterrows():
    print(f"  lag {r['horizon']:6s}  Acc={r['accuracy']:.3f}  Recall={r['recall']:.3f}  ROC-AUC={r['roc_auc']:.3f}")